In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import pynbody

ROMULUS = Path('/mnt/data0/pkrsnak/romulus')
manifest = pd.read_csv(ROMULUS / 'runs' / 'MANIFEST.csv')

print(manifest.shape)
print(manifest.columns.tolist())
manifest[['key', 'lineage', 'layout', 'step', 'real_path', 'unreadable']].head(10)

(52, 17)
['key', 'family', 'variant', 'lineage', 'changa_name', 'step', 'layout', 'real_path', 'owner', 'iBinaryOutput', 'dBHSinkFeedbackEff', 'dBHSinkColdDen', 'dEtaDiffusion', 'aux_format', 'unreadable', 'notes', 'scale_factor']


,key,lineage,layout,step,real_path,unreadable
0,r107,tal-shiar-15,flat,7779,/mnt/data0/pkrsnak/romulus/runs/r107,NaN
1,r142,tal-shiar-15,flat,7779,/mnt/data0/pkrsnak/romulus/runs/r142,NaN
2,r154,tal-shiar-15,flat,7779,/mnt/data0/pkrsnak/romulus/runs/r154,NaN
3,r168,tal-shiar-15,flat,7779,/mnt/data0/pkrsnak/romulus/runs/r168,NaN
4,r204,tal-shiar-15,flat,7779,/mnt/data0/pkrsnak/romulus/runs/r204,NaN
5,r219,tal-shiar-15,flat,7779,/mnt/data0/pkrsnak/romulus/runs/r219,NaN
6,r223,tal-shiar-15,flat,7779,/mnt/data0/pkrsnak/romulus/runs/r223,NaN
7,r239,tal-shiar-15,flat,7779,/mnt/data0/pkrsnak/romulus/runs/r239,NaN
8,r284,tal-shiar-15,flat,7779,/mnt/data0/pkrsnak/romulus/runs/r284,NaN
9,r306,tal-shiar-15,flat,7779,/mnt/data0/pkrsnak/romulus/runs/r306,NaN


In [5]:
def snapshot_path(row):
    run = Path(row.real_path)
    step = str(row.step).zfill(6)
    hits = sorted(run.glob(f'*.{step}')) or sorted(run.glob(f'*.{step}.tipsy'))
    if len(hits) == 1 and hits[0].is_dir():
        hits = [h for h in hits[0].glob(f'*.{step}') if h.is_file()]
    assert len(hits) == 1, (row.key, hits)
    return hits[0]

missing = []
for _, row in manifest.iterrows():
    try:
        p = snapshot_path(row)
        ok = p.exists()
    except AssertionError as e:
        p, ok = e, False
    if not ok:
        missing.append(row.key)
    print(f'{row.key:12s} {row.layout:8s} {p}')

print('\nunresolved:', missing)

r107         flat     /mnt/data0/pkrsnak/romulus/runs/r107/r107.007779.tipsy
r142         flat     /mnt/data0/pkrsnak/romulus/runs/r142/r142.007779.tipsy
r154         flat     /mnt/data0/pkrsnak/romulus/runs/r154/r154.007779.tipsy
r168         flat     /mnt/data0/pkrsnak/romulus/runs/r168/r168.007779.tipsy
r204         flat     /mnt/data0/pkrsnak/romulus/runs/r204/r204.007779.tipsy
r219         flat     /mnt/data0/pkrsnak/romulus/runs/r219/r219.007779.tipsy
r223         flat     /mnt/data0/pkrsnak/romulus/runs/r223/r223.007779.tipsy
r239         flat     /mnt/data0/pkrsnak/romulus/runs/r239/r239.007779.tipsy
r284         flat     /mnt/data0/pkrsnak/romulus/runs/r284/r284.007779.tipsy
r306         flat     /mnt/data0/pkrsnak/romulus/runs/r306/r306.007779.tipsy
r316         flat     /mnt/data0/pkrsnak/romulus/runs/r316/r316.007779.tipsy
r320         flat     /mnt/data0/pkrsnak/romulus/runs/r320/r320.007779.tipsy
r330         flat     /mnt/data0/pkrsnak/romulus/runs/r330/r330.007779.tipsy

In [4]:
row = manifest.set_index('key').loc['r107']
print('changa_name:', row.changa_name)
for p in sorted(Path(row.real_path).iterdir()):
    print(p.name)

changa_name: r107
cosmo25p.768sg1bwK1BHe75_v4.param
r107.007779.tipsy


In [7]:
row = manifest.set_index('key').loc['r618_noBH']
path = snapshot_path(row)

s = pynbody.load(str(path))
s.physical_units()

bh = s.star[s.star['tform'] < 0]
print(path)
print(len(s.star), 'star-family-particles,', len(bh), 'of them BHs')
print(bh['mass'].in_units('Msol'))

/mnt/data0/jillian/romulus_zooms/r618/r618.romulus25.3072g1Hsb.004096
674528 star-family-particles, 0 of them BHs
[]


In [8]:
row = manifest.set_index('key').loc['r618_BH']
path = snapshot_path(row)

s = pynbody.load(str(path))
s.physical_units()

bh = s.star[s.star['tform'] < 0]
print(path)
print(len(s.star), 'star-family-particles,', len(bh), 'of them BHs')
print(bh['mass'].in_units('Msol'))

/mnt/data0/jillian/romulus_zooms/r618_BH/r618.romulus25.3072g1HsbBH.004096
683609 star-family-particles, 1 of them BHs
[739405.99510658]


In [9]:
OUT = ROMULUS / 'vesuvius' / 'products' / 'bh_census'
OUT.mkdir(parents=True, exist_ok=True)

def bh_census(row):
    s = pynbody.load(str(snapshot_path(row)))
    s.physical_units()
    star = s.star
    bh = star[star['tform'] < 0]

    pos = np.asarray(bh['pos'].in_units('kpc'))
    try:                                   # three runs have unreadable .iord, r502_noBH has none
        iord = np.asarray(bh['iord'])
    except Exception:
        iord = np.full(len(bh), -1)

    per_bh = pd.DataFrame({
        'key': row.key,
        'step': int(row.step),
        'iord': iord,
        'mass_Msol': np.asarray(bh['mass'].in_units('Msol')),
        'x_kpc': pos[:, 0], 'y_kpc': pos[:, 1], 'z_kpc': pos[:, 2],
    })
    summary = {
        'key': row.key,
        'step': int(row.step),
        'n_bh': len(bh),
        'total_bh_mass_Msol': float(per_bh.mass_Msol.sum()),
        'max_bh_mass_Msol': float(per_bh.mass_Msol.max()) if len(bh) else 0.0,
        'n_star': len(star) - len(bh),
    }
    return summary, per_bh

summ, tbl = bh_census(manifest[manifest.key == 'r618_BH'].iloc[0])
print(summ)
tbl

{'key': 'r618_BH', 'step': 4096, 'n_bh': 1, 'total_bh_mass_Msol': 739405.995106579, 'max_bh_mass_Msol': 739405.995106579, 'n_star': 683608}


,key,step,iord,mass_Msol,x_kpc,y_kpc,z_kpc
0,r618_BH,4096,20404821,739405.995107,-2208.73151,-2534.588613,-4762.395099


In [10]:
SUMMARY = OUT / 'bh_summary.csv'
done = set(pd.read_csv(SUMMARY).key) if SUMMARY.exists() else set()

for _, row in manifest.iterrows():
    if row.key in done:
        continue
    summ, tbl = bh_census(row)
    tbl.to_csv(OUT / f"{row.key}_{int(row.step):06d}.csv", index=False)
    pd.DataFrame([summ]).to_csv(SUMMARY, mode='a', header=not SUMMARY.exists(), index=False)
    done.add(row.key)
    print(f"{row.key:12s} n_bh={summ['n_bh']:2d}  max={summ['max_bh_mass_Msol']:.3g} Msol")

r107         n_bh= 5  max=2.08e+07 Msol


/mnt/data0/anaconda3/lib/python3.12/site-packages/pynbody/array/__init__.py:505: RuntimeWarning: invalid value encountered in cast
  np.ndarray.__setitem__(self, item, to)


r142         n_bh= 1  max=4.16e+06 Msol
r154         n_bh= 2  max=1.43e+07 Msol
r168         n_bh= 1  max=1.41e+07 Msol
r204         n_bh= 1  max=6.87e+06 Msol
r219         n_bh= 1  max=1.04e+06 Msol
r223         n_bh= 0  max=0 Msol
r239         n_bh= 1  max=2.05e+06 Msol
r284         n_bh= 1  max=6.64e+06 Msol
r306         n_bh= 1  max=1.01e+06 Msol
r316         n_bh= 1  max=3.46e+06 Msol
r320         n_bh= 2  max=2.05e+06 Msol
r330         n_bh= 0  max=0 Msol
r372         n_bh= 0  max=0 Msol
r429         n_bh= 0  max=0 Msol
r431_BH      n_bh= 5  max=5.91e+05 Msol
r431_BH6     n_bh= 6  max=2.71e+05 Msol
r431_noBH    n_bh= 0  max=0 Msol
r488_BH      n_bh= 1  max=1.25e+05 Msol
r488_BH6     n_bh= 1  max=1.3e+05 Msol
r488_BH8     n_bh= 3  max=1.34e+06 Msol
r488_noBH    n_bh= 0  max=0 Msol
r489_BH      n_bh= 3  max=2.33e+05 Msol
r489_noBH    n_bh= 0  max=0 Msol
r502_BH      n_bh= 4  max=1.23e+05 Msol
r502_BH6     n_bh= 2  max=1.2e+05 Msol
r502_noBH    n_bh= 0  max=0 Msol
r515_BH      n_bh=

In [11]:
summary = pd.read_csv(SUMMARY).merge(
    manifest[['key', 'family', 'variant', 'lineage', 'dBHSinkFeedbackEff', 'dEtaDiffusion']],
    on='key')
summary = summary.sort_values(['lineage', 'family', 'variant']).reset_index(drop=True)
summary.to_csv(OUT / 'bh_summary_annotated.csv', index=False)
summary

,key,step,n_bh,total_bh_mass_Msol,max_bh_mass_Msol,n_star,family,variant,lineage,dBHSinkFeedbackEff,dEtaDiffusion
0,r431_BH,4096,5,1.521918e+06,5.906180e+05,2300801,r431,BH,dump-2026-09,0.005,0.30
1,r431_BH6,4096,6,9.267950e+05,2.713358e+05,1019900,r431,BH6,dump-2026-09,0.050,0.30
2,r431_noBH,4096,0,0.000000e+00,0.000000e+00,1659917,r431,noBH,dump-2026-09,NaN,0.15
3,r489_BH,4096,3,4.862857e+05,2.328728e+05,1194848,r489,BH,dump-2026-09,0.005,0.15
4,r489_noBH,4096,0,0.000000e+00,0.000000e+00,1263205,r489,noBH,dump-2026-09,NaN,0.15
5,r502_BH,4096,4,4.443796e+05,1.230756e+05,2161130,r502,BH,dump-2026-09,0.005,0.30
6,r502_BH6,4096,2,2.330298e+05,1.198912e+05,1445474,r502,BH6,dump-2026-09,0.050,0.30
7,r502_noBH,4096,0,0.000000e+00,0.000000e+00,2340481,r502,noBH,dump-2026-09,NaN,0.30
8,r515_BH,4096,1,3.686372e+05,3.686372e+05,1697582,r515,BH,dump-2026-09,0.005,0.15
9,r515_noBH,4096,0,0.000000e+00,0.000000e+00,1150214,r515,noBH,dump-2026-09,NaN,0.15
